In [4]:
"""
build_label_metric.ipynb — scan all bronze JSONL, collect DISTINCT raw metric labels,
and resolve each once via the canonical resolver. Produces a review table.

Why distinct-label, not per-record: there are only a few dozen distinct labels
across the whole portfolio, so you map ~35 things once (and review them), instead
of re-resolving thousands of rows. The map then applies to every record by lookup.

Layered by trust:
  1. dictionary alias  (high)   -> auto-accept
  2. keyword rule      (medium) -> review
  3. unmapped          (none)   -> review (LLM can propose here; human approves)
"""

import json
from pathlib import Path
from collections import defaultdict

from canonical import canonicalize

BRONZE_DIR = Path("output/bronze")



def collect_distinct_labels(min_extraction="high"):
    stats = defaultdict(lambda: {"count": 0, "companies": set()})
    skipped = 0
    for f in BRONZE_DIR.glob("*.jsonl"):
        with open(f, encoding="utf-8") as fh:
            for line in fh:
                rec = json.loads(line)
                ec = rec.get("extraction_confidence", rec.get("confidence", "unknown"))
                if ec != min_extraction:          # only high-extraction records
                    skipped += 1
                    continue
                label = rec["metric"]
                stats[label]["count"] += 1
                stats[label]["companies"].add(rec["company"])
    print(f"collected from high-extraction records; skipped {skipped} low/other")
    return stats


def build_map(stats):
    rows = []
    for label, s in sorted(stats.items(), key=lambda kv: -kv[1]["count"]):
        r = canonicalize(label)
        rows.append({
            "raw_label": label,
            "canonical": r["canonical"],
            "method": r["method"],
            "match_confidence": r["confidence"],
            "count": s["count"],
            "companies": ", ".join(sorted(s["companies"])),
            # auto-accept only high-confidence dictionary hits
            "status": "accepted" if r["confidence"] == "high" else "REVIEW",
        })
    return rows


if __name__ == "__main__":
    stats = collect_distinct_labels()
    rows = build_map(stats)

    print(f"{len(rows)} distinct labels\n")
    print(f"{'raw_label':38s} {'canonical':22s} {'method':9s} {'status':9s} cnt")
    print("-" * 90)
    for r in rows:
        print(f"{r['raw_label']:38s} {str(r['canonical']):22s} "
              f"{r['method']:9s} {r['status']:9s} {r['count']}")

    accepted = sum(1 for r in rows if r["status"] == "accepted")
    print(f"\n{accepted}/{len(rows)} auto-accepted; {len(rows)-accepted} need review")

    # write the map for human review / editing
    out = Path("output/label_metric_map.json")
    out.parent.mkdir(parents=True, exist_ok=True)   # <-- create output/ if missing
    out.write_text(json.dumps(rows, indent=2))
    print(f"wrote {out}")

collected from high-extraction records; skipped 34 low/other
95 distinct labels

raw_label                              canonical              method    status    cnt
------------------------------------------------------------------------------------------
Gross Margin                           gross_margin           alias     accepted  29
Total Headcount                        headcount              alias     accepted  25
Recognized Revenue                     revenue                alias     accepted  12
Monthly Net Burn                       net_burn               alias     accepted  9
Cash Balance                           cash_balance           alias     accepted  9
Net Revenue Retention (LTM)            nrr                    alias     accepted  7
Logo Churn (LTM)                       logo_churn             alias     accepted  7
Contracted ARR                         arr                    alias     accepted  5
Interest Expense                       None                   unmap